# GOES-19 / CPTEC — processamento interativo em cadência nativa

Esta versão usa **todos os NetCDFs disponíveis** no período diurno de cada data, sem impor grade artificial de 10, 30 ou 60 minutos.

Ao concluir cada data, salva os dados em resolução nativa, atualiza a agregação horária e pergunta se pode avançar para o próximo dia.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Execução

Execute a célula abaixo. Para continuar uma data de cada vez, responda `S`; para parar, `N`; e para processar automaticamente o restante, `A`.


In [ ]:
# -*- coding: utf-8 -*-
"""
GOES-19 / CPTEC-INPE — classificação diurna de nuvens em Itajubá
PROCESSAMENTO INTERATIVO POR DATA — CADÊNCIA NATIVA DO PRODUTO

Estratégia:
1) Lê as datas válidas da Base_Consolidada_J11_CSD3_CNR4.xlsx.
2) Para cada data, usa TODOS os NetCDFs disponíveis no intervalo diurno local.
3) Baixa um arquivo por vez.
4) Extrai pixel central + janela 5x5 de Itajubá.
5) Calcula fração de nuvens local.
6) Apaga o NetCDF imediatamente após a extração.
7) Salva:
      - CSV individual de cada data;
      - CSV acumulado em resolução nativa;
      - CSV horário acumulado;
      - CSV resumo diário.
8) Ao terminar uma data, pergunta se deve avançar para a próxima.
9) Se interrompido, retoma automaticamente.

Classes:
- 1, 11 e 16 -> superfície/céu claro ("sup")
- demais classes válidas 1..30 -> nuvem
- 0 / NaN / fora de 1..30 -> sem classificação válida
"""

import sys
import subprocess
import importlib.util
from pathlib import Path
from urllib.parse import urljoin
import re
import time

# ============================================================================
# 0. PACOTES
# ============================================================================
PACOTES = {
    "requests": "requests",
    "bs4": "beautifulsoup4",
    "xarray": "xarray",
    "netCDF4": "netCDF4",
    "pandas": "pandas",
    "numpy": "numpy",
}

faltando = [
    pip_nome
    for modulo, pip_nome in PACOTES.items()
    if importlib.util.find_spec(modulo) is None
]

if faltando:
    print("Instalando pacotes ausentes:", faltando)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *faltando]
    )

import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import xarray as xr


# ============================================================================
# 1. CONFIGURAÇÕES
# ============================================================================

LAT_ITAJUBA = -22.425
LON_ITAJUBA = -45.450
TIMEZONE_LOCAL = "America/Sao_Paulo"

# Base J11-CSD3
NOME_BASE = "Base_Consolidada_J11_CSD3_CNR4.xlsx"
ABA_BASE = "Dados_5min"
USAR_SOMENTE_ALTA_CONFIANCA = True

# Período
ANO = 2026
MESES = [6, 7, 8]

# Janela local de interesse
# Todos os arquivos GOES disponíveis dentro desta janela serão usados.
HORA_LOCAL_INICIO = "07:30"
HORA_LOCAL_FIM = "18:00"

# Janela espacial 5x5
RAIO_PIXELS = 2

# Classes
CLASSES_VALIDAS = set(range(1, 31))
CLASSES_CEU_CLARO = {1, 11, 16}

# Servidor
BASE_REMOTA = (
    "https://ftp.cptec.inpe.br/goes/goes19/"
    "class_nuvens/class_nuvens_diurna_ncf/"
)

# Pastas
PASTA_RAIZ = Path("GOES19_CPTEC_Itajuba_cadencia_nativa")
PASTA_DIAS = PASTA_RAIZ / "dias"
PASTA_TEMP = PASTA_RAIZ / "temp_nc"
PASTA_CATALOGOS = PASTA_RAIZ / "catalogos"

for pasta in [PASTA_RAIZ, PASTA_DIAS, PASTA_TEMP, PASTA_CATALOGOS]:
    pasta.mkdir(parents=True, exist_ok=True)

ARQUIVO_ACUMULADO = PASTA_RAIZ / "GOES19_Itajuba_acumulado_nativo.csv"
ARQUIVO_HORARIO = PASTA_RAIZ / "GOES19_Itajuba_horario.csv"
ARQUIVO_RESUMO = PASTA_RAIZ / "GOES19_Itajuba_resumo_diario.csv"

APAGAR_NC_APOS_EXTRAIR = True


# ============================================================================
# 2. FUNÇÕES DE ACESSO / CATÁLOGO
# ============================================================================

def localizar_base():
    cwd = Path.cwd()

    candidatos = []
    p = cwd

    for _ in range(7):
        candidatos.append(p / NOME_BASE)
        p = p.parent

    candidatos += [
        Path("/content/drive/MyDrive/TCC") / NOME_BASE,
        Path("/content/TCC") / NOME_BASE,
        Path.home() / "TCC" / NOME_BASE,
    ]

    for arq in candidatos:
        if arq.exists():
            return arq

    raizes = [
        cwd,
        cwd.parent,
        Path("/content/drive/MyDrive/TCC"),
        Path("/content/drive/MyDrive"),
        Path.home() / "TCC",
    ]

    encontrados = []

    for raiz in raizes:
        try:
            if raiz.exists():
                encontrados.extend(raiz.rglob(NOME_BASE))
        except (PermissionError, OSError):
            pass

    if encontrados:
        return encontrados[0]

    raise FileNotFoundError(
        f"Não encontrei '{NOME_BASE}'. "
        "Coloque a planilha na mesma pasta do script/notebook."
    )


def listar_links(url, timeout=60):
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    itens = []

    for a in soup.find_all("a", href=True):
        href = a["href"]
        nome = href.rstrip("/").split("/")[-1]

        if nome and nome not in {"..", "Parent Directory"}:
            itens.append({
                "nome": nome,
                "url": urljoin(url, href)
            })

    return pd.DataFrame(itens)


def extrair_timestamp_nome(nome):
    m = re.search(r"(20\d{10})", str(nome))

    if not m:
        return pd.NaT

    return pd.to_datetime(
        m.group(1),
        format="%Y%m%d%H%M",
        utc=True
    )


def catalogo_mes(ano, mes):
    cache = PASTA_CATALOGOS / f"catalogo_{ano}_{mes:02d}.csv"

    if cache.exists():
        cat = pd.read_csv(cache)

        cat["timestamp_utc"] = pd.to_datetime(
            cat["timestamp_utc"], utc=True, errors="coerce"
        )

        cat["timestamp_local"] = pd.to_datetime(
            cat["timestamp_local"], utc=True, errors="coerce"
        ).dt.tz_convert(TIMEZONE_LOCAL)

        cat["data_local"] = cat["timestamp_local"].dt.date

        return cat

    url = f"{BASE_REMOTA}{ano}/{mes:02d}/"

    print("Consultando:", url)

    idx = listar_links(url)

    if idx.empty:
        return pd.DataFrame()

    idx = idx[
        idx["nome"].str.lower().str.endswith(".nc", na=False)
    ].copy()

    idx["timestamp_utc"] = idx["nome"].apply(
        extrair_timestamp_nome
    )

    idx = idx[
        idx["timestamp_utc"].notna()
    ].copy()

    idx["timestamp_local"] = (
        idx["timestamp_utc"].dt.tz_convert(TIMEZONE_LOCAL)
    )

    idx["data_local"] = (
        idx["timestamp_local"].dt.date
    )

    idx = (
        idx
        .sort_values("timestamp_utc")
        .reset_index(drop=True)
    )

    idx.to_csv(cache, index=False)

    return idx


def carregar_datas_validas():
    arquivo_base = localizar_base()

    print("\nBase localizada:")
    print(arquivo_base.resolve())

    df = pd.read_excel(
        arquivo_base,
        sheet_name=ABA_BASE
    )

    if "data_local" not in df.columns:
        raise ValueError("A coluna 'data_local' não existe.")

    df["data_local"] = pd.to_datetime(
        df["data_local"], errors="coerce"
    )

    if "j11_lux" in df.columns:
        df["j11_lux"] = pd.to_numeric(
            df["j11_lux"], errors="coerce"
        )

        df = df[
            df["j11_lux"].notna()
        ].copy()

    if "csd3_minutos_validos_5min" in df.columns:
        df["csd3_minutos_validos_5min"] = pd.to_numeric(
            df["csd3_minutos_validos_5min"],
            errors="coerce"
        )

        df = df[
            df["csd3_minutos_validos_5min"]
            .fillna(0) > 0
        ].copy()

    if (
        USAR_SOMENTE_ALTA_CONFIANCA
        and "nivel_confianca" in df.columns
    ):
        df = df[
            df["nivel_confianca"]
            .astype(str)
            .str.upper()
            .str.contains("ALTA", na=False)
        ].copy()

    mask = (
        (df["data_local"].dt.year == ANO)
        & (df["data_local"].dt.month.isin(MESES))
    )

    datas = sorted(
        df.loc[mask, "data_local"]
        .dt.date
        .dropna()
        .unique()
    )

    return arquivo_base, datas


def construir_catalogo_total():
    partes = []

    for mes in MESES:
        try:
            cat = catalogo_mes(ANO, mes)

            print(
                f"{mes:02d}/{ANO}: "
                f"{len(cat)} arquivos catalogados."
            )

            if len(cat):
                partes.append(cat)

        except Exception as e:
            print(
                f"Falha ao catalogar {mes:02d}/{ANO}:",
                repr(e)
            )

    if not partes:
        raise RuntimeError("Nenhum catálogo foi carregado.")

    return pd.concat(
        partes,
        ignore_index=True
    )


def arquivos_da_data(catalogo, data_local):
    """
    Retorna TODOS os arquivos disponíveis da data no intervalo local.
    Não cria grade artificial de 10, 30 ou 60 min.
    """
    cat = catalogo[
        catalogo["data_local"] == data_local
    ].copy()

    if cat.empty:
        return cat

    ini_h, ini_m = map(int, HORA_LOCAL_INICIO.split(":"))
    fim_h, fim_m = map(int, HORA_LOCAL_FIM.split(":"))

    ini_min = ini_h * 60 + ini_m
    fim_min = fim_h * 60 + fim_m

    minuto_dia = (
        cat["timestamp_local"].dt.hour * 60
        + cat["timestamp_local"].dt.minute
    )

    cat = cat[
        (minuto_dia >= ini_min)
        & (minuto_dia <= fim_min)
    ].copy()

    return (
        cat
        .sort_values("timestamp_utc")
        .reset_index(drop=True)
    )


# ============================================================================
# 3. DOWNLOAD / EXTRAÇÃO
# ============================================================================

def baixar_com_retry(url, destino, tentativas=4, timeout=180):
    destino = Path(destino)

    for tentativa in range(1, tentativas + 1):
        try:
            with requests.get(
                url,
                stream=True,
                timeout=timeout
            ) as r:
                r.raise_for_status()

                with open(destino, "wb") as f:
                    for chunk in r.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            f.write(chunk)

            return destino

        except Exception as e:
            print(
                f"    Falha {tentativa}/{tentativas}: "
                f"{repr(e)}"
            )

            if destino.exists():
                destino.unlink(missing_ok=True)

            if tentativa < tentativas:
                time.sleep(3 * tentativa)

    raise RuntimeError(
        f"Falha definitiva no download: {url}"
    )


def obter_indices_itajuba(ds):
    lat = np.asarray(ds["lat"].values)
    lon = np.asarray(ds["lon"].values)

    iy = int(
        np.nanargmin(
            np.abs(lat - LAT_ITAJUBA)
        )
    )

    ix = int(
        np.nanargmin(
            np.abs(lon - LON_ITAJUBA)
        )
    )

    return (
        iy,
        ix,
        float(lat[iy]),
        float(lon[ix])
    )


def extrair_janela(caminho_nc, timestamp_utc):
    with xr.open_dataset(caminho_nc) as ds:

        if "Band1" not in ds.data_vars:
            raise ValueError("Band1 não encontrada.")

        if "lat" not in ds.coords or "lon" not in ds.coords:
            raise ValueError("Coordenadas lat/lon não encontradas.")

        iy, ix, lat_pix, lon_pix = obter_indices_itajuba(ds)

        r = RAIO_PIXELS

        y0 = max(0, iy - r)
        y1 = min(ds.sizes["lat"], iy + r + 1)

        x0 = max(0, ix - r)
        x1 = min(ds.sizes["lon"], ix + r + 1)

        janela = np.asarray(
            ds["Band1"]
            .isel(
                lat=slice(y0, y1),
                lon=slice(x0, x1)
            )
            .values
        )

        central = float(
            ds["Band1"]
            .isel(lat=iy, lon=ix)
            .values
        )

    arred = np.rint(janela)

    mask_valida = (
        np.isfinite(arred)
        & np.isin(
            arred,
            list(CLASSES_VALIDAS)
        )
    )

    validos = (
        arred[mask_valida]
        .astype(int)
    )

    n_total = int(janela.size)
    n_validos = int(validos.size)

    if n_validos > 0:

        mask_claro = np.isin(
            validos,
            list(CLASSES_CEU_CLARO)
        )

        n_claros = int(
            mask_claro.sum()
        )

        n_nuvem = int(
            n_validos - n_claros
        )

        fracao_claro = (
            n_claros / n_validos
        )

        fracao_nuvem = (
            n_nuvem / n_validos
        )

        classes, contagens = np.unique(
            validos,
            return_counts=True
        )

        k = int(
            np.argmax(contagens)
        )

        classe_moda = int(
            classes[k]
        )

        fracao_moda = float(
            contagens[k] / contagens.sum()
        )

    else:

        n_claros = 0
        n_nuvem = 0
        fracao_claro = np.nan
        fracao_nuvem = np.nan
        classe_moda = np.nan
        fracao_moda = np.nan

    if (
        np.isfinite(central)
        and int(round(central)) in CLASSES_VALIDAS
    ):

        classe_central = int(
            round(central)
        )

        central_valido = True

        central_nuvem = int(
            classe_central
            not in CLASSES_CEU_CLARO
        )

    else:

        classe_central = np.nan
        central_valido = False
        central_nuvem = np.nan

    ts_utc = pd.Timestamp(
        timestamp_utc
    )

    ts_local = ts_utc.tz_convert(
        TIMEZONE_LOCAL
    )

    return {
        "timestamp_utc": ts_utc,
        "timestamp_local": ts_local,
        "data_local": str(ts_local.date()),
        "hora_local": ts_local.strftime("%H:%M"),

        "classe_central": classe_central,
        "central_valido": central_valido,
        "central_nuvem": central_nuvem,

        "classe_moda_5x5": classe_moda,
        "fracao_moda_5x5": fracao_moda,

        "n_pixels_total_5x5": n_total,
        "n_pixels_validos_5x5": n_validos,
        "n_pixels_claros_5x5": n_claros,
        "n_pixels_nuvem_5x5": n_nuvem,

        "fracao_valida_5x5": (
            n_validos / n_total
            if n_total else np.nan
        ),

        "fracao_ceu_claro_5x5": fracao_claro,
        "fracao_nuvem_5x5": fracao_nuvem,

        "lat_pixel": lat_pix,
        "lon_pixel": lon_pix,
    }


# ============================================================================
# 4. SALVAMENTO / AGREGAÇÃO
# ============================================================================

def caminho_dia(data_local):
    return (
        PASTA_DIAS
        / f"GOES19_Itajuba_{pd.Timestamp(data_local):%Y%m%d}.csv"
    )


def carregar_dia_existente(data_local):
    arq = caminho_dia(data_local)

    if not arq.exists():
        return pd.DataFrame()

    d = pd.read_csv(arq)

    d["timestamp_utc"] = pd.to_datetime(
        d["timestamp_utc"],
        utc=True,
        errors="coerce"
    )

    return d


def salvar_dia(df, data_local):
    arq = caminho_dia(data_local)

    d = df.copy()

    d["timestamp_utc"] = pd.to_datetime(
        d["timestamp_utc"],
        utc=True,
        errors="coerce"
    )

    d = (
        d
        .drop_duplicates(
            subset=["timestamp_utc"]
        )
        .sort_values("timestamp_utc")
        .reset_index(drop=True)
    )

    d.to_csv(
        arq,
        index=False
    )

    return arq


def atualizar_produtos_acumulados():
    arquivos = sorted(
        PASTA_DIAS.glob(
            "GOES19_Itajuba_*.csv"
        )
    )

    if not arquivos:
        return (
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame()
        )

    partes = []

    for arq in arquivos:
        try:
            partes.append(
                pd.read_csv(arq)
            )
        except Exception as e:
            print(
                "Aviso ao ler",
                arq.name,
                ":",
                repr(e)
            )

    if not partes:
        return (
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame()
        )

    acum = pd.concat(
        partes,
        ignore_index=True
    )

    acum["timestamp_utc"] = pd.to_datetime(
        acum["timestamp_utc"],
        utc=True,
        errors="coerce"
    )

    acum["timestamp_local"] = pd.to_datetime(
        acum["timestamp_local"],
        utc=True,
        errors="coerce"
    ).dt.tz_convert(
        TIMEZONE_LOCAL
    )

    acum = (
        acum
        .drop_duplicates(
            subset=["timestamp_utc"]
        )
        .sort_values("timestamp_utc")
        .reset_index(drop=True)
    )

    acum.to_csv(
        ARQUIVO_ACUMULADO,
        index=False
    )

    # ---------------- HORÁRIO ----------------
    acum["inicio_hora_local"] = (
        acum["timestamp_local"]
        .dt.floor("h")
    )

    horario = (
        acum
        .groupby(
            "inicio_hora_local",
            as_index=False
        )
        .agg(
            n_imagens=("fracao_nuvem_5x5", "count"),

            fcloud_media=("fracao_nuvem_5x5", "mean"),
            fcloud_mediana=("fracao_nuvem_5x5", "median"),
            fcloud_min=("fracao_nuvem_5x5", "min"),
            fcloud_max=("fracao_nuvem_5x5", "max"),
            fcloud_dp=("fracao_nuvem_5x5", "std"),

            fracao_pixel_central_nuvem=(
                "central_nuvem",
                "mean"
            ),

            fracao_valida_media=(
                "fracao_valida_5x5",
                "mean"
            ),
        )
    )

    horario["data_local"] = (
        horario["inicio_hora_local"]
        .dt.date
        .astype(str)
    )

    horario.to_csv(
        ARQUIVO_HORARIO,
        index=False
    )

    # ---------------- DIÁRIO ----------------
    resumo = (
        acum
        .groupby(
            "data_local",
            as_index=False
        )
        .agg(
            n_imagens=("fracao_nuvem_5x5", "count"),

            fcloud_media=("fracao_nuvem_5x5", "mean"),
            fcloud_mediana=("fracao_nuvem_5x5", "median"),
            fcloud_min=("fracao_nuvem_5x5", "min"),
            fcloud_max=("fracao_nuvem_5x5", "max"),

            fracao_pixel_central_nuvem=(
                "central_nuvem",
                "mean"
            ),
        )
    )

    resumo.to_csv(
        ARQUIVO_RESUMO,
        index=False
    )

    return acum, horario, resumo


def datas_concluidas():
    concluidas = set()

    for arq in PASTA_DIAS.glob(
        "GOES19_Itajuba_*.csv"
    ):

        m = re.search(
            r"(\d{8})",
            arq.name
        )

        if m:
            try:
                concluidas.add(
                    pd.to_datetime(
                        m.group(1),
                        format="%Y%m%d"
                    ).date()
                )
            except Exception:
                pass

    return concluidas


# ============================================================================
# 5. PROCESSAMENTO DE UMA DATA
# ============================================================================

def processar_data(data_local, catalogo):

    print("\n" + "=" * 80)
    print("PROCESSANDO DATA:", data_local)
    print("=" * 80)

    plano = arquivos_da_data(
        catalogo,
        data_local
    )

    if plano.empty:
        print(
            "Nenhum NetCDF disponível na janela diurna."
        )
        return False

    # Diagnóstico da cadência REAL
    tempos = plano["timestamp_local"]

    if len(tempos) >= 2:
        intervalos = (
            tempos.diff()
            .dropna()
            .dt.total_seconds()
            / 60
        )

        cad_mediana = (
            float(intervalos.median())
            if len(intervalos)
            else np.nan
        )
    else:
        cad_mediana = np.nan

    print(
        f"Arquivos disponíveis: {len(plano)}"
    )

    print(
        f"Cadência mediana observada: "
        f"{cad_mediana:.1f} min"
        if pd.notna(cad_mediana)
        else "Cadência mediana: não calculável"
    )

    print(
        f"Primeiro horário local: "
        f"{plano['timestamp_local'].min()}"
    )

    print(
        f"Último horário local : "
        f"{plano['timestamp_local'].max()}"
    )

    existente = carregar_dia_existente(
        data_local
    )

    if len(existente):

        processados = set(
            existente["timestamp_utc"]
            .dropna()
            .astype(str)
        )

        linhas = existente.to_dict(
            "records"
        )

        print(
            f"Retomada: {len(existente)} arquivos "
            f"já estavam salvos."
        )

    else:

        processados = set()
        linhas = []

    total = len(plano)

    for pos, (_, row) in enumerate(
        plano.iterrows(),
        start=1
    ):

        ts = pd.Timestamp(
            row["timestamp_utc"]
        )

        chave = str(ts)

        if chave in processados:

            print(
                f"  [{pos:03d}/{total:03d}] "
                f"{row['timestamp_local']} — já processado"
            )

            continue

        destino = (
            PASTA_TEMP
            / row["nome"]
        )

        print(
            f"  [{pos:03d}/{total:03d}] "
            f"{row['timestamp_local']}"
        )

        try:

            baixar_com_retry(
                row["url"],
                destino
            )

            resultado = extrair_janela(
                destino,
                ts
            )

            resultado["arquivo"] = row["nome"]
            resultado["url"] = row["url"]

            linhas.append(
                resultado
            )

            parcial = pd.DataFrame(
                linhas
            )

            salvar_dia(
                parcial,
                data_local
            )

            processados.add(
                chave
            )

            fc = resultado[
                "fracao_nuvem_5x5"
            ]

            if pd.isna(fc):

                print(
                    "       Fcloud = sem classificação válida"
                )

            else:

                print(
                    f"       classe central = "
                    f"{resultado['classe_central']} | "
                    f"Fcloud 5x5 = {fc:.2f}"
                )

        except Exception as e:

            print(
                "       ERRO:",
                repr(e)
            )

        finally:

            if (
                APAGAR_NC_APOS_EXTRAIR
                and destino.exists()
            ):
                try:
                    destino.unlink()
                except Exception:
                    pass

    df_dia = pd.DataFrame(
        linhas
    )

    if df_dia.empty:
        print(
            "Nenhuma observação válida foi extraída."
        )
        return False

    arq_dia = salvar_dia(
        df_dia,
        data_local
    )

    acum, horario, resumo = (
        atualizar_produtos_acumulados()
    )

    linha_resumo = resumo[
        resumo["data_local"]
        .astype(str)
        == str(data_local)
    ]

    print("\nDATA CONCLUÍDA")
    print("-" * 80)

    print(
        "Arquivo diário:",
        arq_dia.resolve()
    )

    print(
        "Acumulado nativo:",
        ARQUIVO_ACUMULADO.resolve()
    )

    print(
        "Agregado horário:",
        ARQUIVO_HORARIO.resolve()
    )

    print(
        "Resumo diário:",
        ARQUIVO_RESUMO.resolve()
    )

    if len(linha_resumo):

        r = linha_resumo.iloc[0]

        print(
            f"Nº de imagens válidas : "
            f"{int(r['n_imagens'])}"
        )

        print(
            f"Fcloud média          : "
            f"{r['fcloud_media']:.3f}"
        )

        print(
            f"Fcloud mediana        : "
            f"{r['fcloud_mediana']:.3f}"
        )

        print(
            f"Fcloud min/máx        : "
            f"{r['fcloud_min']:.3f} / "
            f"{r['fcloud_max']:.3f}"
        )

    # Mostra também o agregado horário do dia
    if len(horario):

        h_dia = horario[
            horario["data_local"]
            .astype(str)
            == str(data_local)
        ].copy()

        if len(h_dia):
            print("\nAGREGADO HORÁRIO DO DIA")
            print(
                h_dia[
                    [
                        "inicio_hora_local",
                        "n_imagens",
                        "fcloud_media",
                        "fcloud_mediana",
                        "fcloud_min",
                        "fcloud_max"
                    ]
                ].to_string(
                    index=False
                )
            )

    return True


# ============================================================================
# 6. MAIN INTERATIVO
# ============================================================================

def main():

    arquivo_base, datas_validas = (
        carregar_datas_validas()
    )

    if not datas_validas:
        raise RuntimeError(
            "Nenhuma data válida foi encontrada."
        )

    print(
        "\nDatas válidas:",
        len(datas_validas)
    )

    print(
        "Primeira:",
        datas_validas[0]
    )

    print(
        "Última  :",
        datas_validas[-1]
    )

    print(
        "\nConstruindo/carregando catálogo remoto..."
    )

    catalogo = construir_catalogo_total()

    catalogo = catalogo[
        catalogo["data_local"]
        .isin(set(datas_validas))
    ].copy()

    concluidas = datas_concluidas()

    pendentes = [
        d
        for d in datas_validas
        if d not in concluidas
    ]

    print(
        "\nDatas já concluídas:",
        len(concluidas)
    )

    print(
        "Datas pendentes:",
        len(pendentes)
    )

    if not pendentes:

        print(
            "\nTodas as datas válidas já foram processadas."
        )

        atualizar_produtos_acumulados()

        return

    print(
        "\nPrimeira data pendente:",
        pendentes[0]
    )

    for i, data in enumerate(
        pendentes
    ):

        if i > 0:

            print(
                "\n" + "-" * 80
            )

            resposta = input(
                f"Podemos passar para a próxima data, {data}? "
                "[S]im / [N]ão / [A]utomático até o fim: "
            ).strip().lower()

            if resposta in {
                "n", "nao", "não", ""
            }:

                print(
                    "\nProcessamento encerrado. "
                    "Tudo que já foi concluído está salvo."
                )

                break

            if resposta in {
                "a", "auto",
                "automatico", "automático"
            }:

                print(
                    "\nModo automático ativado."
                )

                for data_auto in pendentes[i:]:

                    processar_data(
                        data_auto,
                        catalogo
                    )

                print(
                    "\nTodas as datas pendentes foram percorridas."
                )

                break

            if resposta not in {
                "s", "sim", "y", "yes"
            }:

                print(
                    "Resposta não reconhecida. "
                    "Encerrando."
                )

                break

        sucesso = processar_data(
            data,
            catalogo
        )

        if not sucesso:

            print(
                "\nA data não foi concluída. "
                "Na próxima execução, o programa tentará novamente."
            )

    print(
        "\n" + "=" * 80
    )

    print(
        "FIM DA EXECUÇÃO"
    )

    print(
        "=" * 80
    )

    if ARQUIVO_ACUMULADO.exists():
        print(
            "Acumulado:",
            ARQUIVO_ACUMULADO.resolve()
        )

    if ARQUIVO_HORARIO.exists():
        print(
            "Horário:",
            ARQUIVO_HORARIO.resolve()
        )

    if ARQUIVO_RESUMO.exists():
        print(
            "Resumo:",
            ARQUIVO_RESUMO.resolve()
        )


if __name__ == "__main__":
    main()
